# Notebook 12: Matched ensemble control and frozen test check

Select a **CPU runtime** and run the **single executable cell**. It reads the existing full Notebook 09 and Notebook 11 folders, including saved auxiliary action caches and controller models. Keep those folders in `MyDrive/SensorRepair_ICDSIS`. There are no dataset downloads or manual uploads, and no PyTorch/GPU requirement.

**Scope:** DSADS and HAR, fold 0, seed 2026 only. It reproduces all four Notebook 11 validation decisions exactly before continuing. It preserves the earlier policies and all old outputs.

**Matched control:** Three F1-utility/harm head pairs are fitted per dataset, for **12 small CPU head fits** in total. Every head keeps all source participants and randomly subsamples original windows within each participant. All fault views/actions of each selected window remain together. Each pair matches the corresponding source-jackknife pair's training-row count, nominal model capacity, iterations and model random seed. Subsets and F1 targets use source data only. The same six fitted heads provide mean and minimum controls; validation selects their thresholds using the original grid and harm guards. Head count and row count are matched, while exact training time is measured rather than assumed equal.

**Before opening any test prediction arrays**, the notebook completes both datasets' validation checks and control selection, then writes a frozen evaluation plan. `source_mean_f1` is the primary candidate on both datasets, based on the prior validation comparison. No dataset-specific test winner is selected. All 22 earlier methods, all four Notebook 11 variants and both matched controls are retained.

**Evaluation:** Existing candidate probabilities are reused. Only small saved controllers produce new test actions. Clean, known faults, and stuck faults excluded from fitting are reported separately. Outputs include per-condition and per-participant results, harmful changes relative to keep, worst-condition harm, and actual class-decision changes. Sensor-action intervention rates are available for the six action controllers; they are distinct from class changes and do not demonstrate neural inference savings.

These are **exploratory tests after prior test inspection**, not a new untouched confirmation. Stuck was excluded from model fitting, but its earlier test results were inspected. This notebook has one fold and one seed per dataset, makes no significance claims or risk guarantees, and does not report confidence intervals. Participant diagnostics use classes present within that participant and are separate from the primary all-class Macro-F1.

The 60-minute session budget pauses between head fits/phases. Rerun the same cell to resume. Completed head fits are reused; a head interrupted before saving must be fitted again. Do not change settings midway. A scikit-learn version matching Notebook 11 is required for its saved models.

Outputs: `SensorRepair_ICDSIS/outputs/notebook12_v1`.

Share the final printed output, `notebook12_test_summary.csv`, `notebook12_comparisons.csv`, `notebook12_harm_audit.csv`, and `notebook12_run_report.json`. The full control audit and validation/per-participant tables are also saved.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import hashlib
import json
import os
import shutil
import time
import zlib
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

PROJECT=Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
OUTPUT=PROJECT/'outputs'/'notebook12_v1'
LOCAL=Path('/content/sensorrepair_notebook12_v1')
CODE_VERSION='sensorrepair_nb12_v1'
CELL_SIGNATURE='04d8e84d460e8b15d8ec242af060b37cafbbadfcc0ceef3c3e0524048fcb28db'
SESSION_MINUTES=60
JOBS=[{'dataset':d,'fold':'fold_0','seed':2026} for d in ['dsads','har']]
DEV_CONDITIONS=['clean']+[f'{f}_{i}' for f in ['dropout','bias','gain'] for i in [1,2,3]]
CONDITIONS=DEV_CONDITIONS+[f'stuck_{i}' for i in [1,2,3]]
SCOPES={'clean':['clean'],'known_faults':DEV_CONDITIONS[1:],'stuck_excluded_from_fitting':CONDITIONS[10:]}
ORIGINAL_FAMILIES=['oof_accuracy_utility','oof_f1_utility','source_mean_f1','source_min_f1']
CONTROL_FAMILIES=['window_mean_f1','window_min_f1']
NEW_METHODS=ORIGINAL_FAMILIES+CONTROL_FAMILIES
OLD_METHODS=['keep','mask_suspect','reconstruct_suspect','residual_gate_mask','residual_gate_reconstruct',
 'confidence_select','oracle_diagnostic','harm_aware_policy','benefit_only_policy','confidence_margin_tuned',
 'residual_mask_tuned','residual_reconstruct_tuned','ternary_outcome_selector','utility_regression_selector',
 'uniform_fusion_safe','uniform_fusion_accuracy','global_fusion_safe','global_fusion_accuracy',
 'gated_fusion_safe','gated_fusion_accuracy','residual_stacking_safe','residual_stacking_accuracy']
METHODS=OLD_METHODS+NEW_METHODS
PRIMARY='source_mean_f1'
METRICS=['macro_f1','accuracy','decision_change_rate','helpful_rate','harmful_rate']
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
SESSION_DEADLINE=time.monotonic()+SESSION_MINUTES*60
print('NOTEBOOK 12 | matched controller ensemble and frozen exploratory test check | CPU | one executable cell',flush=True)
print('Both datasets, fold_0, seed 2026. No CNN/reconstructor fitting or signal-model inference.',flush=True)
print('Phase A: reproduce validation, fit small matched controls, freeze both datasets. Phase B: cached test evaluation.',flush=True)
print('Previous test outcomes were inspected. These comparisons remain exploratory.',flush=True)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def atomic_joblib(path, value):
    temporary = LOCAL / (Path(path).name + '.tmp')
    joblib.dump(value, temporary, compress=3)
    partial = Path(path).with_name(Path(path).name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)


def evaluate_choices(pack, choices, method):
    p, y = pack['probabilities'], pack['y']
    D, N, A, K = p.shape
    assert choices.shape == (D, N) and choices.min() >= 0 and choices.max() < A
    predictions = p.argmax(axis=-1)
    original_correct = predictions[:, :, 0] == y[None, :]
    selected = np.take_along_axis(predictions, choices[:, :, None], axis=2).squeeze(2)
    rows = []
    for i, condition in enumerate(pack['conditions']):
        correct = selected[i] == y
        helpful = float((~original_correct[i] & correct).mean())
        harmful = float((original_correct[i] & ~correct).mean())
        accuracy = float(correct.mean())
        assert abs(accuracy - original_correct[i].mean() - helpful + harmful) < 1e-10
        rows.append({'condition': condition, 'method': method, 'windows': N,
                     'macro_f1': macro_f1(y, selected[i], K), 'accuracy': accuracy,
                     'intervention_rate': float((choices[i] != 0).mean()), 'helpful_rate': helpful,
                     'harmful_rate': harmful,
                     'harm_given_original_correct': float(np.sum(original_correct[i] & ~correct) / max(1, original_correct[i].sum()))})
    return rows


def aggregate_rows(rows):
    clean = next(row for row in rows if row['condition'] == 'clean')
    corrupted = [row for row in rows if row['condition'] != 'clean']
    answer = {key: float(np.mean([row[key] for row in corrupted])) for key in
              ['macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']}
    answer.update(clean_macro_f1=clean['macro_f1'], clean_accuracy=clean['accuracy'],
                  clean_harmful_rate=clean['harmful_rate'], clean_intervention_rate=clean['intervention_rate'])
    return answer


class PauseStudy(Exception):
    pass

def probability_statistics(probability):
    sorted_p = np.sort(probability, axis=-1)
    confidence = sorted_p[..., -1]
    margin = confidence - sorted_p[..., -2]
    entropy = -(probability * np.log(np.clip(probability, 1e-7, 1.0))).sum(axis=-1)
    return np.stack([confidence, margin, entropy], axis=-1)


def policy_features(probabilities, residuals, statistics, residual_scale):
    # This function accepts observable arrays only. It cannot access labels, subjects,
    # source IDs, injected fault locations, or condition/severity metadata.
    D, N, A, K = probabilities.shape
    G, J, M = residuals.shape[-1], A - 1, D * N
    p = probabilities.reshape(M, A, K)
    r = residuals.reshape(M, G)
    s = statistics.reshape(M, G, 4)
    original, candidates = p[:, 0], p[:, 1:]
    candidate_groups = np.tile(np.arange(G), 2)
    original_statistics = probability_statistics(original)
    candidate_statistics = probability_statistics(candidates)
    scaled = r / residual_scale[None, :]

    def repeat(values):
        return np.broadcast_to(values[:, None, :], (M, J, values.shape[-1]))

    middle = (original[:, None, :] + candidates) / 2.0
    js = 0.5 * (original[:, None, :] * np.log(np.clip(original[:, None, :], 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    js += 0.5 * (candidates * np.log(np.clip(candidates, 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    names, pieces = [], []

    def add(values, labels):
        assert values.shape == (M, J, len(labels))
        pieces.append(values)
        names.extend(labels)

    add(repeat(original), [f'original_probability_{k}' for k in range(K)])
    add(candidates, [f'action_probability_{k}' for k in range(K)])
    add(candidates - original[:, None, :], [f'probability_difference_{k}' for k in range(K)])
    add(repeat(original_statistics), ['original_confidence', 'original_margin', 'original_entropy'])
    add(candidate_statistics, ['action_confidence', 'action_margin', 'action_entropy'])
    add(candidate_statistics - original_statistics[:, None, :], ['confidence_difference', 'margin_difference', 'entropy_difference'])
    add(js[..., None], ['prediction_js_divergence'])
    add((candidates.argmax(axis=-1) == original.argmax(axis=-1)[:, None])[..., None], ['prediction_agreement'])
    kinds = np.zeros((J, 2), dtype=np.float32)
    kinds[:G, 0], kinds[G:, 1] = 1.0, 1.0
    add(np.broadcast_to(kinds[None], (M, J, 2)), ['action_is_mask', 'action_is_reconstruction'])
    add(np.broadcast_to(np.eye(G, dtype=np.float32)[candidate_groups][None], (M, J, G)), [f'candidate_group_{g}' for g in range(G)])
    add(repeat(np.log1p(r)), [f'log_residual_group_{g}' for g in range(G)])
    add(np.log1p(r[:, candidate_groups])[..., None], ['candidate_log_residual'])
    add(np.log1p(scaled[:, candidate_groups])[..., None], ['candidate_log_scaled_residual'])
    add(repeat(np.log1p(scaled.max(axis=-1, keepdims=True))), ['maximum_log_scaled_residual'])
    add(s[:, candidate_groups, :], ['candidate_mean_absolute_signal', 'candidate_signal_std', 'candidate_temporal_difference', 'candidate_zero_fraction'])
    add(repeat(s.reshape(M, G * 4)), [f'observed_group_{g}_stat_{j}' for g in range(G) for j in range(4)])
    values = np.ascontiguousarray(np.concatenate(pieces, axis=-1).reshape(M * J, -1), dtype=np.float32)
    assert len(set(names)) == len(names) and np.isfinite(values).all()
    return values, names


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def pack_from_caches(paths, expected_signature):
    matrices, residuals, statistics = [], [], []
    anchor, conditions = None, []
    for path in paths:
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == expected_signature
            identity = {key: saved[key] for key in ['y', 'subjects', 'window_ids', 'source_indices', 'action_names']}
            if anchor is None:
                anchor = identity
            else:
                for key, value in identity.items():
                    assert np.array_equal(value, anchor[key]), f'Cache alignment error: {key}'
            matrices.append(saved['probabilities'])
            residuals.append(saved['reconstruction_residuals'])
            statistics.append(saved['observed_group_statistics'])
            conditions.append(str(saved['condition'].item()))
    pack = dict(anchor)
    pack.update(probabilities=np.stack(matrices), residuals=np.stack(residuals),
                statistics=np.stack(statistics), conditions=conditions)
    p, r, s = pack['probabilities'], pack['residuals'], pack['statistics']
    D, N, A, K = p.shape
    G = r.shape[-1]
    assert A == 1 + 2 * G and r.shape == (D, N, G) and s.shape == (D, N, G, 4)
    assert np.isfinite(p).all() and np.isfinite(r).all() and np.isfinite(s).all()
    assert np.all(p >= 0) and np.allclose(p.sum(axis=-1), 1.0, atol=1e-5)
    assert np.all(r >= 0) and len(np.unique(pack['window_ids'])) == N
    return pack


def identity(job):
    return f'{job["dataset"]}_{job["fold"]}_seed_{job["seed"]}'


def assert_table_parity(computed,original,keys,columns):
    assert not computed.duplicated(keys).any() and not original.duplicated(keys).any()
    a=computed.set_index(keys).sort_index()
    b=original.set_index(keys).sort_index()
    assert a.index.equals(b.index),'Table row identities changed.'
    for col in columns:
        assert np.allclose(a[col].to_numpy(dtype=float),b[col].to_numpy(dtype=float),atol=1e-12,rtol=0,equal_nan=True),f'Metric parity failed: {col}'

def verified_copy(source,relative,expected):
    assert not Path(relative).is_absolute() and '..' not in Path(relative).parts
    target=LOCAL/'inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_file() and file_sha256(target)==expected:return target
    partial=target.with_name(target.name+'.partial')
    shutil.copyfile(source,partial)
    assert file_sha256(partial)==expected,f'Input hash mismatch: {source}'
    os.replace(partial,target)
    return target


def stage_json(number,name):
    root=PROJECT/'outputs'/f'notebook{number:02d}_v1'
    inv={r['filename']:r for r in json.loads((root/f'notebook{number:02d}_artifact_inventory.json').read_text())}
    path=verified_copy(root/name,f'nb{number:02d}/{name}',inv[name]['sha256'])
    return json.loads(path.read_text()),inv[name]['sha256']


def subject_weights(subjects):
    people,inv,counts=np.unique(subjects,return_inverse=True,return_counts=True)
    return (1./counts[inv]).astype(np.float64)


def counterfactual_targets(predictions,y,subjects,classes):
    # Target is exact for ONE replaced prediction in a fixed keep confusion
    # matrix. It is a marginal utility surrogate, not additive/global F1.
    D,N,A=predictions.shape;J=A-1
    w=subject_weights(subjects);total=w.sum()
    weights=np.broadcast_to(w[None,:,None],(D,N,J)).copy()
    keep=predictions[:,:,0];actions=predictions[:,:,1:]
    correct=predictions==y[None,:,None]
    fix=(~correct[:,:,:1]&correct[:,:,1:]).astype(np.float32)
    harm=(correct[:,:,:1]&~correct[:,:,1:]).astype(np.float32)
    utility=np.zeros((D,N,J),dtype=np.float64)
    for d in range(D):
        C=np.bincount(y*classes+keep[d],weights=w,minlength=classes*classes).reshape(classes,classes)
        denominator=C.sum(0)+C.sum(1);tp=C.diagonal()
        old=np.divide(2.*tp,denominator,out=np.zeros(classes),where=denominator>0)
        b=np.broadcast_to(keep[d,:,None],(N,J));a=actions[d]
        wb=np.broadcast_to(w[:,None],(N,J))
        db=np.maximum(denominator[b]-wb,0.);da=denominator[a]+wb
        tpb=tp[b]-wb*(y[:,None]==b);tpa=tp[a]+wb*(y[:,None]==a)
        fb=np.divide(2.*tpb,db,out=np.zeros_like(db),where=db>1e-12)
        fa=np.divide(2.*tpa,da,out=np.zeros_like(da),where=da>1e-12)
        utility[d]=((fb-old[b])+(fa-old[a]))/classes*total/wb
        utility[d][a==b]=0.
    assert np.isfinite(utility).all()
    # Normalize sample weights to mean 1, preserving equal total mass per person.
    weights/=weights.mean()
    return utility.astype(np.float32),fix,harm,weights.astype(np.float32)


def head_prediction(model,X,kind):
    if isinstance(model,dict):return np.full(len(X),model['constant'],dtype=np.float32)
    with threadpool_limits(limits=2):
        if kind=='binary':return model.predict_proba(X)[:,list(model.classes_).index(1)].astype(np.float32)
        return model.predict(X).astype(np.float32)


def scored_actions(scores,threshold):
    if threshold is None:return np.zeros(scores.shape[:2],dtype=np.int64)
    return np.where(scores.max(-1)>threshold,scores.argmax(-1)+1,0)


def select_family(pack,scores_by_penalty,family):
    base=aggregate_rows(evaluate_choices(pack,np.zeros(scores_by_penalty[1.].shape[:2],dtype=np.int64),'keep'))
    best=None;trials=[]
    for penalty in HARM_PENALTIES:
        for threshold in THRESHOLDS:
            actions=scored_actions(scores_by_penalty[penalty],threshold)
            summary=aggregate_rows(evaluate_choices(pack,actions,family))
            feasible=(summary['clean_macro_f1']>=base['clean_macro_f1']-CLEAN_F1_TOLERANCE-1e-12
                and summary['clean_harmful_rate']<=MAX_HARM+1e-12 and summary['harmful_rate']<=MAX_HARM+1e-12)
            trials.append({'family':family,'harm_penalty':penalty,'threshold':threshold,'feasible':feasible,**summary})
            rank=(summary['macro_f1'],summary['clean_macro_f1'],-summary['harmful_rate'],-summary['intervention_rate'],threshold is None)
            if feasible and (best is None or rank>best['rank']):
                best={'rank':rank,'settings':{'family':family,'harm_penalty':penalty,'threshold':threshold},
                      'summary':summary,'actions':actions,'structural_keep_only':threshold is None}
    assert best is not None
    del best['rank']
    return best,trials

def metrics(y,pred,base,K,present=None):
    cm=np.bincount(y*K+pred,minlength=K*K).reshape(K,K)
    den=cm.sum(0)+cm.sum(1)
    f1=np.divide(2.*cm.diagonal(),den,out=np.zeros(K),where=den>0)
    correct=pred==y;was_correct=base==y
    helpful=float((~was_correct & correct).mean());harmful=float((was_correct & ~correct).mean())
    assert abs(correct.mean()-was_correct.mean()-helpful+harmful)<1e-12
    return {'macro_f1':float(f1.mean() if present is None else f1[present].mean()),'accuracy':float(correct.mean()),
            'decision_change_rate':float((pred!=base).mean()),'helpful_rate':helpful,'harmful_rate':harmful}

def check_budget():
    if time.monotonic()>=SESSION_DEADLINE:
        raise PauseStudy('Session budget reached; finished head fits and phases are saved.')


SOURCE11=PROJECT/'outputs/notebook11_v1'
SOURCE9=PROJECT/'outputs/notebook09_v1'
STAGES={f'nb{i:02d}':PROJECT/'outputs'/f'notebook{i:02d}_v1' for i in [2,4,5]}
report11,report11_sha=stage_json(11,'notebook11_run_report.json')
manifest11,manifest11_sha=stage_json(11,'notebook11_study_manifest.json')
report9,report9_sha=stage_json(9,'notebook09_run_report.json')
manifest9,manifest9_sha=stage_json(9,'notebook09_study_manifest.json')
assert report11['study_complete'] and report11['configurations_complete']==2,'Finish Notebook 11 first.'
assert report9['study_complete'] and report9['configurations_complete']==15,'Finish Notebook 09 first.'
assert report11['study_signature']==manifest11['signature']==json_signature(manifest11['plan'])
assert report9['study_signature']==manifest9['signature']==json_signature(manifest9['plan'])
assert manifest11['plan']['source_manifest_sha256']==manifest9_sha
expected_version=manifest11['plan']['software']['sklearn']
assert sklearn.__version__==expected_version,(
    f'Saved controllers require scikit-learn {expected_version}; this runtime has {sklearn.__version__}. '
    'Use the same scikit-learn version as Notebook 11 before running this cell.')
MAX_ITER=manifest11['plan']['controller_head_iterations']
HARM_PENALTIES=manifest11['plan']['validation_grid']['harm_penalty']
THRESHOLDS=manifest11['plan']['validation_grid']['threshold']
CLEAN_F1_TOLERANCE=manifest11['plan']['validation_guards']['clean_f1_loss']
MAX_HARM=manifest11['plan']['validation_guards']['clean_and_mean_corrupt_harm']
assert HARM_PENALTIES==[1.,2.,4.] and THRESHOLDS[-1] is None
RESULTS11={identity(r):r for r in report11['completed_results']}
RESULTS9={identity(r):r for r in report9['completed_results']}
CACHE_INPUTS={identity(j):manifest9['plan']['cache_inputs'][identity(j)] for j in JOBS}
assert set(RESULTS11)=={identity(j) for j in JOBS}
AUXILIARY_INPUTS={}
for job in JOBS:
    key=identity(job)
    assert manifest11['plan']['cache_inputs'][key]=={k:v for k,v in CACHE_INPUTS[key].items() if k.split('|')[0] in ['policy','validation']}
    entries=[]
    for audit in RESULTS11[key]['source_audit']:
        if audit['cohort']=='original_policy':continue
        relative=f'runs/{key}/auxiliary/cohort_{int(audit["cohort"]):02d}'
        assert json.loads((SOURCE11/relative/'source_split.json').read_text())==audit
        for condition in DEV_CONDITIONS:
            name=relative+'/'+condition+'.npz'
            digest=json.loads((SOURCE11/(name+'.sha256.json')).read_text())
            assert digest['signature']==audit['signature']
            entries.append({'filename':name,'condition':condition,'cohort':audit['cohort'],**digest})
    assert len(entries)==RESULTS11[key]['auxiliary_pairs']*len(DEV_CONDITIONS)
    AUXILIARY_INPUTS[key]=entries
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'jobs':JOBS,
 'source11_report_sha256':report11_sha,'source11_manifest_sha256':manifest11_sha,
 'source9_report_sha256':report9_sha,'source9_manifest_sha256':manifest9_sha,
 'cache_inputs':CACHE_INPUTS,'auxiliary_cache_inputs':AUXILIARY_INPUTS,
 'primary_method':PRIMARY,'primary_choice':'same family for both datasets, chosen from Notebook 11 validation',
 'new_control_families':CONTROL_FAMILIES,'control_head_pairs_per_dataset':3,
 'control_subset':'stratified random original-window subsamples without replacement; every source person retained; all condition/action views kept together',
 'matched_properties':['head count','training action-row counts','nominal HGB capacity','iterations','model random seeds'],
 'control_targets':'recompute participant-balanced exact scaled single-replacement F1 and harm on each selected source subset',
 'validation_grid':{'harm_penalty':HARM_PENALTIES,'threshold':THRESHOLDS},
 'validation_guards':{'clean_f1_loss':CLEAN_F1_TOLERANCE,'clean_and_mean_corrupt_harm':MAX_HARM},
 'phase_order':'freeze both validation selections before any test prediction array is opened',
 'metrics':METRICS,'scopes':SCOPES,'methods':METHODS,
 'primary_metric':'mean of condition-level all-class Macro-F1 within each scope; one fold and one seed per dataset',
 'participant_metric':'mean condition-level Macro-F1 over classes present within each participant; secondary diagnostics only',
 'test_status':'exploratory after prior test inspection; stuck excluded from fitting, earlier stuck outcomes inspected',
 'jackknife_limitation':report11['jackknife_limitation'],
 'confidence_intervals_or_p_values':False,'new_signal_training_or_inference':False,
 'software':{'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__}}
STUDY_SIGNATURE=json_signature(plan)
path=OUTPUT/'notebook12_study_manifest.json'
if path.exists():
    assert json.loads(path.read_text())['signature']==STUDY_SIGNATURE,'Study inputs, code or environment changed; keep the frozen configuration.'
else:save_json(path,{'signature':STUDY_SIGNATURE,'plan':plan})
print('Study plan recorded:',STUDY_SIGNATURE[:16],flush=True)


def source_run_file(stage,job,name):
    result=(RESULTS11 if stage==11 else RESULTS9)[identity(job)]
    record=next(a for a in result['artifacts'] if a['filename']==name)
    relative='runs/'+identity(job)+'/'+name
    return verified_copy((SOURCE11 if stage==11 else SOURCE9)/relative,f'nb{stage}/'+relative,record['sha256'])


def load_pack(job,split):
    assert split in ['policy','validation','test']
    if split=='test':
        assert (OUTPUT/'notebook12_frozen_test_plan.json').exists(),'Both datasets must be frozen before reading test arrays.'
    conditions=CONDITIONS if split=='test' else DEV_CONDITIONS
    paths=[];signature=None
    for condition in conditions:
        item=CACHE_INPUTS[identity(job)][split+'|'+condition]
        if signature is None:signature=item['signature']
        assert signature==item['signature']
        path=verified_copy(STAGES[item['stage']]/item['filename'],item['stage']+'/'+item['filename'],item['sha256'])
        with np.load(path,allow_pickle=False) as z:
            assert z['condition'].item()==condition and z['split'].item()==split
        paths.append(path)
    pack=pack_from_caches(paths,signature)
    assert pack['conditions']==conditions
    return pack


def pooled_source_pack(job):
    packs=[load_pack(job,'policy')]
    entries=AUXILIARY_INPUTS[identity(job)]
    for cohort in sorted({a['cohort'] for a in entries}):
        items=[a for a in entries if a['cohort']==cohort]
        assert [a['condition'] for a in items]==DEV_CONDITIONS
        signature=items[0]['signature'];assert all(a['signature']==signature for a in items)
        paths=[verified_copy(SOURCE11/a['filename'],'nb11/'+a['filename'],a['sha256']) for a in items]
        p=pack_from_caches(paths,signature)
        assert p['conditions']==DEV_CONDITIONS
        packs.append(p)
    first=packs[0]
    assert all(np.array_equal(p['action_names'],first['action_names']) for p in packs)
    out={k:np.concatenate([p[k] for p in packs]) for k in ['y','subjects','window_ids','source_indices']}
    out.update({k:np.concatenate([p[k] for p in packs],axis=1) for k in ['probabilities','residuals','statistics']})
    out.update(conditions=DEV_CONDITIONS,action_names=first['action_names'])
    assert len(np.unique(out['window_ids']))==len(np.unique(out['source_indices']))==len(out['y'])
    assert np.unique(out['subjects']).tolist()==RESULTS11[identity(job)]['source_participants']
    assert len(out['y'])==RESULTS11[identity(job)]['source_windows']
    return out


def feature_matrix(pack,configuration):
    scale=np.asarray(configuration['residual_scale'],dtype=np.float32)
    X,names=policy_features(pack['probabilities'],pack['residuals'],pack['statistics'],scale)
    assert names==configuration['feature_names']
    return X


def original_head(job,name,configuration):
    payload=joblib.load(source_run_file(11,job,name+'.joblib'))
    assert payload['signature']==configuration['signature']
    model=payload['model']
    if not isinstance(model,dict):assert model.get_params()['max_iter']==MAX_ITER
    return model


def original_scores(job,pack,configuration):
    X=feature_matrix(pack,configuration)
    D,N,A,K=pack['probabilities'].shape;shape=(D,N,A-1)
    def pred(name,kind):return head_prediction(original_head(job,name,configuration),X,kind).reshape(shape)
    pf=pred('pooled_fix','binary');ph=pred('pooled_harm','binary');pu=pred('pooled_f1','regression')
    su=np.stack([pred(f'jackknife_{b}_f1','regression') for b in range(3)])
    sh=np.stack([pred(f'jackknife_{b}_harm','binary') for b in range(3)])
    scores={f:{} for f in ORIGINAL_FAMILIES}
    for penalty in HARM_PENALTIES:
        scores['oof_accuracy_utility'][penalty]=pf-penalty*ph
        scores['oof_f1_utility'][penalty]=pu-(penalty-1.)*ph
        s=su-(penalty-1.)*sh
        scores['source_mean_f1'][penalty]=s.mean(0)
        scores['source_min_f1'][penalty]=s.min(0)
    return scores


def stratified_window_subset(subjects,count,seed):
    people,counts=np.unique(subjects,return_counts=True)
    assert len(people)<=count<=len(subjects)
    expected=counts.astype(float)*count/len(subjects)
    allocations=np.minimum(counts,np.maximum(np.floor(expected).astype(int),1))
    while allocations.sum()<count:
        eligible=np.flatnonzero(allocations<counts)
        pick=eligible[np.argmax((expected-allocations)[eligible])];allocations[pick]+=1
    while allocations.sum()>count:
        eligible=np.flatnonzero(allocations>1)
        pick=eligible[np.argmax((allocations-expected)[eligible])];allocations[pick]-=1
    rng=np.random.default_rng(seed);mask=np.zeros(len(subjects),dtype=bool)
    for person,n in zip(people,allocations):
        candidates=np.flatnonzero(subjects==person)
        mask[rng.choice(candidates,size=int(n),replace=False)]=True
    assert int(mask.sum())==count and np.array_equal(np.unique(subjects[mask]),people)
    return mask


def control_mask(job,pack,block,run):
    D,N,A,K=pack['probabilities'].shape;J=A-1
    matched=next(a for a in RESULTS11[identity(job)]['jackknife_audit'] if a['block']==block)
    rows=matched['training_rows'];assert rows%(D*J)==0
    count=int(rows//(D*J))
    seed=2026+zlib.crc32((identity(job)+'|window-control|'+str(block)).encode())
    mask=stratified_window_subset(pack['subjects'],count,seed)
    audit={'dataset':job['dataset'],'fold':job['fold'],'seed':job['seed'],'block':block,
      'source_participants':np.unique(pack['subjects']).tolist(),
      'selected_participants':np.unique(pack['subjects'][mask]).tolist(),
      'source_windows':N,'selected_windows':count,'action_rows':int(count*D*J),
      'matched_source_jackknife_action_rows':rows,'head_iterations':MAX_ITER,
      'subset_seed':int(seed),'selection_unit':'original window; all condition/action views retained',
      'window_ids_sha256':json_signature(pack['window_ids'][mask].tolist()),
      'selected_counts_by_participant':{str(int(s)):int(np.sum(pack['subjects'][mask]==s)) for s in np.unique(pack['subjects'])}}
    signature=json_signature({'study':STUDY_SIGNATURE,**audit});audit['signature']=signature
    path=run/f'window_{block}_subset.npz'
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            assert z['signature'].item()==signature and np.array_equal(z['mask'],mask)
    else:save_npz(path,signature=np.asarray(signature),mask=mask,source_indices=pack['source_indices'][mask],window_ids=pack['window_ids'][mask])
    return mask,audit


def fit_matched_head(block,kind,X,target,weight,run,audit):
    suffix='f1' if kind=='regression' else 'harm'
    name=f'window_{block}_{suffix}';path=run/(name+'.joblib')
    seed=int(2026+zlib.crc32(f'jackknife_{block}_{suffix}'.encode())%10000)
    if path.exists():
        saved=joblib.load(path)
        assert saved['signature']==audit['signature'] and saved['random_state']==seed
        assert saved['training_rows']==audit['action_rows']
        print('  matched head',name,': reused.',flush=True)
        return saved['model']
    check_budget();start=time.perf_counter()
    if np.ptp(target)==0:model={'constant':float(target[0])}
    else:
        cls=HistGradientBoostingRegressor if kind=='regression' else HistGradientBoostingClassifier
        model=cls(learning_rate=.05,max_iter=MAX_ITER,max_leaf_nodes=15,max_depth=4,
          min_samples_leaf=40,l2_regularization=5.,max_bins=64,categorical_features=None,
          early_stopping=False,random_state=seed)
        with threadpool_limits(limits=2):model.fit(X,target,sample_weight=weight)
    seconds=time.perf_counter()-start
    atomic_joblib(path,{'signature':audit['signature'],'model':model,'kind':kind,'random_state':seed,
      'training_rows':len(target),'training_participants':audit['selected_participants'],'training_seconds':seconds})
    print(f'  matched head {name}: {len(target)} rows | {seconds:.1f}s',flush=True)
    return model


def control_scores(pack,configuration,run,audits):
    X=feature_matrix(pack,configuration);D,N,A,K=pack['probabilities'].shape
    utility=[];harm=[]
    for b in range(3):
        audit=next(a for a in audits if a['block']==b)
        def pred(suffix,kind):
            payload=joblib.load(run/f'window_{b}_{suffix}.joblib')
            assert payload['signature']==audit['signature']
            return head_prediction(payload['model'],X,kind).reshape(D,N,A-1)
        utility.append(pred('f1','regression'));harm.append(pred('harm','binary'))
    utility=np.stack(utility);harm=np.stack(harm)
    result={f:{} for f in CONTROL_FAMILIES}
    for penalty in HARM_PENALTIES:
        s=utility-(penalty-1.)*harm
        result['window_mean_f1'][penalty]=s.mean(0)
        result['window_min_f1'][penalty]=s.min(0)
    return result


def assert_saved_artifacts(run,result):
    assert result['study_signature']==STUDY_SIGNATURE
    for item in result['artifacts']:assert file_sha256(run/item['filename'])==item['sha256'],item['filename']


def local_artifacts(run,names):
    return [{'filename':name,'sha256':file_sha256(run/name)} for name in names]


def phase_a(job):
    run=OUTPUT/'runs'/identity(job);run.mkdir(parents=True,exist_ok=True)
    done=run/'validation_complete.json'
    if done.exists():
        r=json.loads(done.read_text());assert_saved_artifacts(run,r)
        print('  Reused frozen validation:',identity(job),flush=True)
        return r
    check_budget();print('\nPHASE A',identity(job),flush=True)
    configuration=json.loads(source_run_file(11,job,'controller_inputs.json').read_text())
    frozen_original=json.loads(source_run_file(11,job,'selected_policies.json').read_text())
    assert frozen_original['signature']==configuration['signature']
    assert frozen_original['selected_methods']==RESULTS11[identity(job)]['selected_methods']
    validation=load_pack(job,'validation')
    assert np.unique(validation['subjects']).tolist()==configuration['validation_participants']
    scores=original_scores(job,validation,configuration)
    selected=dict(frozen_original['selected_methods'])
    reproduced=[]
    with np.load(source_run_file(11,job,'validation_decisions.npz'),allow_pickle=False) as z:
        assert z['signature'].item()==configuration['signature'] and z['methods'].tolist()==ORIGINAL_FAMILIES
        for key in ['y','subjects','window_ids','source_indices']:assert np.array_equal(z[key],validation[key])
        for i,family in enumerate(ORIGINAL_FAMILIES):
            settings=selected[family]['settings']
            actions=scored_actions(scores[family][settings['harm_penalty']],settings['threshold'])
            assert np.array_equal(actions,z['actions'][i]),'Notebook 11 validation decisions did not reproduce.'
            summary=aggregate_rows(evaluate_choices(validation,actions,family))
            for k,value in selected[family]['summary'].items():assert abs(summary[k]-value)<1e-12,(family,k)
            reproduced.append(actions)
    print('  Notebook 11 validation decisions and metrics reproduced exactly.',flush=True)
    del scores
    source=pooled_source_pack(job)
    assert not np.intersect1d(source['subjects'],validation['subjects']).size
    assert not np.intersect1d(source['window_ids'],validation['window_ids']).size
    X=feature_matrix(source,configuration);predictions=source['probabilities'].argmax(-1)
    D,N,A,K=source['probabilities'].shape;J=A-1;audits=[]
    for block in range(3):
        mask,audit=control_mask(job,source,block,run);audits.append(audit)
        t,_,h,w=counterfactual_targets(predictions[:,mask],source['y'][mask],source['subjects'][mask],K)
        use=np.broadcast_to(mask[None,:,None],(D,N,J)).reshape(-1)
        subX=X[use]
        fit_matched_head(block,'regression',subX,t.reshape(-1),w.reshape(-1),run,audit)
        fit_matched_head(block,'binary',subX,h.reshape(-1),w.reshape(-1),run,audit)
        del subX,t,h,w
    save_json(run/'matched_control_audit.json',audits)
    control=control_scores(validation,configuration,run,audits)
    trials=[];control_rows=[]
    for family in CONTROL_FAMILIES:
        best,grid=select_family(validation,control[family],family)
        reproduced.append(best.pop('actions'));selected[family]=best;trials.extend(grid)
        control_rows.append({**job,'method':family,'source':'matched window ensemble validation selection',
          'structural_keep_only':best['structural_keep_only'],**best['summary']})
        print('  Selected',family,':',best['settings'],flush=True)
    oldtable=pd.read_csv(source_run_file(11,job,'validation_summary.csv'))
    save_csv(run/'validation_summary.csv',pd.concat([oldtable,pd.DataFrame(control_rows)],ignore_index=True))
    save_csv(run/'control_validation_trials.csv',pd.DataFrame(trials))
    save_npz(run/'validation_decisions.npz',methods=np.asarray(NEW_METHODS),actions=np.stack(reproduced),
       conditions=np.asarray(DEV_CONDITIONS),y=validation['y'],subjects=validation['subjects'],
       window_ids=validation['window_ids'],source_indices=validation['source_indices'])
    selection={'study_signature':STUDY_SIGNATURE,'job':job,'configuration':configuration,
      'selected_methods':selected,'matched_control_audit':audits,'primary_method':PRIMARY}
    save_json(run/'frozen_policies.json',selection)
    names=['validation_summary.csv','control_validation_trials.csv','validation_decisions.npz',
       'matched_control_audit.json','frozen_policies.json']
    names += [f'window_{b}_{suffix}.joblib' for b in range(3) for suffix in ['f1','harm']]
    names += [f'window_{b}_subset.npz' for b in range(3)]
    result={**job,'study_signature':STUDY_SIGNATURE,'validation_parity_passed':True,
      'validation_windows':len(validation['y']),'validation_participants':np.unique(validation['subjects']).tolist(),
      'selected_methods':selected,'matched_control_audit':audits,'artifacts':local_artifacts(run,names)}
    save_json(done,result)
    del X,predictions,source,validation,control
    gc.collect()
    return result


def condition_rows(job,y,predictions,conditions,classes,actions=None):
    base=predictions[0];rows=[]
    assert predictions.shape==(len(METHODS),len(conditions),len(y))
    for m,method in enumerate(METHODS):
        for d,condition in enumerate(conditions):
            row={**job,'method':method,'condition':condition,'windows':len(y),**metrics(y,predictions[m,d],base[d],classes)}
            row['sensor_action_intervention_rate']=(float((actions[NEW_METHODS.index(method),d]!=0).mean())
                 if actions is not None and method in NEW_METHODS else (0. if method=='keep' else np.nan))
            rows.append(row)
    return pd.DataFrame(rows)


def participant_rows(job,y,subjects,predictions,classes):
    rows=[]
    for m,method in enumerate(METHODS):
        if method=='oracle_diagnostic':continue
        for person in np.unique(subjects):
            use=subjects==person;present=np.unique(y[use])
            for scope,conditions in SCOPES.items():
                values=[metrics(y[use],predictions[m,CONDITIONS.index(c),use],predictions[0,CONDITIONS.index(c),use],classes,present) for c in conditions]
                average={k:float(np.mean([v[k] for v in values])) for k in METRICS}
                average['macro_f1_present_classes']=average.pop('macro_f1')
                rows.append({**job,'method':method,'subject':int(person),'scope':scope,'windows':int(use.sum()),
                    'classes_present':len(present),**average})
    return pd.DataFrame(rows)


def phase_b(job,selection):
    run=OUTPUT/'runs'/identity(job);done=run/'test_complete.json'
    if done.exists():
        r=json.loads(done.read_text());assert_saved_artifacts(run,r)
        print('  Reused test evaluation:',identity(job),flush=True)
        return r
    check_budget();print('\nPHASE B',identity(job),flush=True)
    assert_saved_artifacts(run,selection)
    frozen=json.loads((run/'frozen_policies.json').read_text())
    pack=load_pack(job,'test');configuration=frozen['configuration']
    assert not np.intersect1d(pack['subjects'],configuration['source_participants']+configuration['validation_participants']).size
    scores=original_scores(job,pack,configuration)
    scores.update(control_scores(pack,configuration,run,frozen['matched_control_audit']))
    actions=[]
    for family in NEW_METHODS:
        item=frozen['selected_methods'][family];settings=item['settings']
        a=scored_actions(scores[family][settings['harm_penalty']],settings['threshold'])
        if item['structural_keep_only']:assert not a.any()
        actions.append(a)
    actions=np.stack(actions)
    candidates=pack['probabilities'].argmax(-1)
    new_pred=np.stack([np.take_along_axis(candidates,a[:,:,None],axis=2).squeeze(2) for a in actions])
    with np.load(source_run_file(9,job,'test_predictions.npz'),allow_pickle=False) as z:
        assert z['signature'].item()==RESULTS9[identity(job)]['signature']
        assert z['methods'].tolist()==OLD_METHODS and z['conditions'].tolist()==CONDITIONS
        for key in ['y','subjects','window_ids','source_indices']:assert np.array_equal(z[key],pack[key])
        old_pred=z['predictions']
    assert np.array_equal(old_pred[0],candidates[:,:,0]),'Keep predictions differ between saved baseline and candidate caches.'
    predictions=np.concatenate([old_pred,new_pred])
    K=pack['probabilities'].shape[-1]
    assert predictions.min()>=0 and predictions.max()<K
    rows=condition_rows(job,pack['y'],predictions,CONDITIONS,K,actions)
    original=pd.read_csv(source_run_file(9,job,'per_condition.csv'))
    assert_table_parity(rows[rows.method.isin(OLD_METHODS)],original,
       ['dataset','fold','seed','method','condition'],METRICS)
    participants=participant_rows(job,pack['y'],pack['subjects'],predictions,K)
    summaries=[]
    for scope,conditions in SCOPES.items():
        frame=rows[rows.condition.isin(conditions)].groupby(['dataset','fold','seed','method'],sort=False)[METRICS+['sensor_action_intervention_rate']].mean().reset_index()
        frame['scope']=scope;summaries.append(frame)
    summary=pd.concat(summaries,ignore_index=True)
    save_csv(run/'per_condition.csv',rows);save_csv(run/'per_participant.csv',participants);save_csv(run/'test_summary.csv',summary)
    save_npz(run/'test_predictions.npz',study_signature=np.asarray(STUDY_SIGNATURE),methods=np.asarray(METHODS),
      conditions=np.asarray(CONDITIONS),predictions=predictions,new_action_methods=np.asarray(NEW_METHODS),
      new_actions=actions,y=pack['y'],subjects=pack['subjects'],window_ids=pack['window_ids'],source_indices=pack['source_indices'])
    result={**job,'study_signature':STUDY_SIGNATURE,'test_windows':len(pack['y']),
      'test_participants':np.unique(pack['subjects']).tolist(),'original_test_metric_parity_passed':True,
      'original_threshold_changes':0,'matched_control_thresholds_selected_on':'validation only',
      'artifacts':local_artifacts(run,['per_condition.csv','per_participant.csv','test_summary.csv','test_predictions.npz'])}
    save_json(done,result)
    print(f'  Reproduced all 22 previous methods; evaluated all six action controllers on {len(pack["y"])} test windows.',flush=True)
    del scores,pack,predictions,new_pred
    gc.collect()
    return result


def build_audits(summary,conditions,participants):
    comparisons=[];harm=[]
    for (dataset,fold,seed,scope),group in summary.groupby(['dataset','fold','seed','scope'],sort=False):
        lookup={r['method']:r for r in group.to_dict('records')}
        for focal in [PRIMARY,'source_min_f1','oof_f1_utility']:
            for comparator,row in lookup.items():
                if comparator in [focal,'oracle_diagnostic']:continue
                comparisons.append({'dataset':dataset,'fold':fold,'seed':seed,'scope':scope,'method':focal,
                  'comparator':comparator,'primary_method':focal==PRIMARY,
                  'policy_macro_f1_pct':lookup[focal]['macro_f1']*100,'comparator_macro_f1_pct':row['macro_f1']*100,
                  'delta_macro_f1_pp':(lookup[focal]['macro_f1']-row['macro_f1'])*100,
                  'policy_harmful_rate_pct':lookup[focal]['harmful_rate']*100,
                  'comparator_harmful_rate_pct':row['harmful_rate']*100})
        for method,row in lookup.items():
            part=conditions[(conditions.dataset==dataset)&(conditions.method==method)&conditions.condition.isin(SCOPES[scope])]
            people=participants[(participants.dataset==dataset)&(participants.method==method)&(participants.scope==scope)]
            record={'dataset':dataset,'fold':fold,'seed':seed,'scope':scope,'method':method,
              'macro_f1_pct':row['macro_f1']*100,'mean_harm_pct':row['harmful_rate']*100,
              'mean_harm_le_1pct':bool(row['harmful_rate']<=.01+1e-12),
              'maximum_condition_harm_pct':float(part.harmful_rate.max()*100),
              'conditions_exceeding_1pct_harm':int((part.harmful_rate>.01+1e-12).sum()),
              'worst_condition_macro_f1_pct':float(part.macro_f1.min()*100),
              'maximum_participant_mean_harm_pct':float(people.harmful_rate.max()*100) if len(people) else np.nan,
              'participants_exceeding_1pct_mean_harm':int((people.harmful_rate>.01+1e-12).sum()) if len(people) else np.nan,
              'test_participants':len(people),'descriptive_only':True}
            if scope=='clean':
                loss=(lookup['keep']['macro_f1']-row['macro_f1'])*100
                record.update(clean_f1_loss_vs_keep_pp=loss,clean_f1_loss_le_0_5pp=bool(loss<=.5+1e-12))
            harm.append(record)
    return pd.DataFrame(comparisons),pd.DataFrame(harm)


phase_a_results=[];phase_b_results=[];paused=None
try:
    for job in JOBS:phase_a_results.append(phase_a(job))
    freeze={'study_signature':STUDY_SIGNATURE,'primary_method':PRIMARY,'all_methods':METHODS,
       'validation_complete':{identity(j):file_sha256(OUTPUT/'runs'/identity(j)/'validation_complete.json') for j in JOBS},
       'frozen_policy_hashes':{identity(j):file_sha256(OUTPUT/'runs'/identity(j)/'frozen_policies.json') for j in JOBS}}
    frozen_path=OUTPUT/'notebook12_frozen_test_plan.json'
    if frozen_path.exists():assert json.loads(frozen_path.read_text())==freeze,'Frozen validation selections changed.'
    else:save_json(frozen_path,freeze)
    print('\nBoth datasets frozen before test access:',json_signature(freeze)[:16],flush=True)
    for job,r in zip(JOBS,phase_a_results):phase_b_results.append(phase_b(job,r))
except PauseStudy as e:
    paused=str(e);print('\nPAUSED:',paused,'Rerun the same cell.',flush=True)

report={'code_version':CODE_VERSION,'study_signature':STUDY_SIGNATURE,
 'study_complete':len(phase_b_results)==len(JOBS),'validation_configurations_complete':len(phase_a_results),
 'test_configurations_complete':len(phase_b_results),'configurations_required':len(JOBS),
 'validation_results':phase_a_results,'test_results':phase_b_results,'pause_reason':paused,
 'primary_method':PRIMARY,'methods_reported':len(METHODS),'new_cpu_heads_required':12,
 'new_signal_model_training':0,'new_signal_model_inference':0,'new_controller_test_decisions':True,
 'previous_notebook11_threshold_changes':0,'test_based_threshold_selection':False,
 'original_outputs_modified':False,'test_status':plan['test_status'],
 'confidence_intervals_or_p_values':False,'harm_or_novelty_guarantee':False,
 'coverage':'one fold and one seed per dataset; DSADS has two test participants, HAR has nine in the original full data',
 'jackknife_limitation':plan['jackknife_limitation']}
save_json(OUTPUT/'notebook12_run_report.json',report)
if phase_a_results:
    tables=[pd.read_csv(OUTPUT/'runs'/identity(r)/'validation_summary.csv') for r in phase_a_results]
    save_csv(OUTPUT/'notebook12_validation_summary.csv',pd.concat(tables,ignore_index=True))
    audits=[]
    for r in phase_a_results:
        for a in r['matched_control_audit']:
            row=dict(a)
            for key in ['source_participants','selected_participants','selected_counts_by_participant']:row[key]=json.dumps(row[key])
            audits.append(row)
    save_csv(OUTPUT/'notebook12_matched_control_audit.csv',pd.DataFrame(audits))
if phase_b_results:
    def merged(name):return pd.concat([pd.read_csv(OUTPUT/'runs'/identity(r)/name) for r in phase_b_results],ignore_index=True)
    summary=merged('test_summary.csv');conditions=merged('per_condition.csv');participants=merged('per_participant.csv')
    comparisons,harm=build_audits(summary,conditions,participants)
    for name,table in [('test_summary',summary),('per_condition',conditions),('per_participant',participants),
                       ('comparisons',comparisons),('harm_audit',harm)]:save_csv(OUTPUT/f'notebook12_{name}.csv',table)
    view=comparisons[(comparisons.method==PRIMARY)&comparisons.comparator.isin(
       ['keep','harm_aware_policy','confidence_select','residual_stacking_safe','window_mean_f1'])]
    print('\nPRIMARY EXPLORATORY TEST COMPARISONS | percentages; deltas in percentage points',flush=True)
    print(view[['dataset','scope','comparator','policy_macro_f1_pct','comparator_macro_f1_pct','delta_macro_f1_pp',
        'policy_harmful_rate_pct']].round(3).to_string(index=False),flush=True)
    selected_harm=harm[harm.method.isin([PRIMARY,'source_min_f1','window_mean_f1','window_min_f1'])]
    print('\nOBSERVED HARM AUDIT | not guarantees',flush=True)
    print(selected_harm[['dataset','scope','method','mean_harm_pct','maximum_condition_harm_pct',
        'conditions_exceeding_1pct_harm','maximum_participant_mean_harm_pct']].round(3).to_string(index=False),flush=True)
print('\nNOTEBOOK 12','COMPLETE' if report['study_complete'] else 'PAUSED',flush=True)
print(f'Validation configurations: {len(phase_a_results)}/2 | Test configurations: {len(phase_b_results)}/2 | Signal training/inference: 0',flush=True)
print('One fold and one seed per dataset. No test-based tuning, confidence intervals, significance or risk guarantees.',flush=True)
print('Outputs:',OUTPUT,flush=True)
print('Share final output, notebook12_test_summary.csv, notebook12_comparisons.csv, notebook12_harm_audit.csv, notebook12_run_report.json.',flush=True)
records=[{'filename':p.name,'sha256':file_sha256(p),'bytes':p.stat().st_size}
         for p in sorted(OUTPUT.iterdir()) if p.is_file() and p.name!='notebook12_artifact_inventory.json']
save_json(OUTPUT/'notebook12_artifact_inventory.json',records)


Mounted at /content/drive
NOTEBOOK 12 | matched controller ensemble and frozen exploratory test check | CPU | one executable cell
Both datasets, fold_0, seed 2026. No CNN/reconstructor fitting or signal-model inference.
Phase A: reproduce validation, fit small matched controls, freeze both datasets. Phase B: cached test evaluation.
Previous test outcomes were inspected. These comparisons remain exploratory.
Study plan recorded: bf5b3bfdb153bc75

PHASE A dsads_fold_0_seed_2026
  Notebook 11 validation decisions and metrics reproduced exactly.
  matched head window_0_f1: 342000 rows | 29.1s
  matched head window_0_harm: 342000 rows | 30.1s
  matched head window_1_f1: 342000 rows | 29.5s
  matched head window_1_harm: 342000 rows | 30.4s
  matched head window_2_f1: 456000 rows | 37.8s
  matched head window_2_harm: 456000 rows | 40.1s
  Selected window_mean_f1 : {'family': 'window_mean_f1', 'harm_penalty': 2.0, 'threshold': 0.1}
  Selected window_min_f1 : {'family': 'window_min_f1', 'harm_p